# Data Retrieval


In [ ]:
from google.colab import drive
import os
import requests
from rich.progress import track
import pandas as pd

In [ ]:
# Mount drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Check if dataset exist
DATA_DIR = '/content/drive/MyDrive/CMS_PRO-Explainable_Clustering/data'
DATASET_DIR = 'geodata_dataset.csv' # Dataset file name

dataset_path = os.path.join(DATA_DIR, DATASET_DIR)

In [ ]:
def fetch_data(query='*'):
    base_url = "https://onestop4all.nfdi4earth.de/fastapi"

    params = {
        "q": query,
        "fq": "type:\"http://www.w3.org/ns/dcat#Dataset\"",
        "rows": 5000,
        "start": 0,
        "fl": "*, [child author]",
        "ident": "true",
        "q.op": "OR",
        "defType": "edismax",
        "bq": "isEdutrain:true^1000",
        "qf": "title^1 keyword^50 collector",
        "facet": "true",
        "facet.field": [
            "type",
            "subjectArea_str",
            "dataAccessType",
            "contentType_str",
            "dataUploadType",
            "supportsMetadataStandard_str",
            "software_license_str",
            "assignsIdentifierScheme"
        ],
        "facet.range": "datePublished",
        "facet.range.start": "2000-01-01T00:00:00Z",
        "facet.range.end": "2024-01-01T00:00:00Z",
        "facet.range.gap": "+1YEAR"
    }

    all_docs = []
    total_docs = 0

    try:
        res = requests.get(base_url, params=params)
        res.raise_for_status()
        data = res.json()
        total_docs = data.get("response", {}).get("numFound", 0)

        if total_docs == 0:
            print("No documents found.")
            return []

        all_docs.extend(data.get("response", {}).get("docs", []))

        for start in track(
            range(5000, total_docs, 5000),
            description="Fetching documents"
        ):
            params["start"] = start
            res = requests.get(base_url, params=params)
            res.raise_for_status()
            data = res.json()
            all_docs.extend(data.get("response", {}).get("docs", []))

        return all_docs

    except requests.exceptions.RequestException as e:
        print(f"Error: {e}")
        return []

In [ ]:
def save_data_to_csv(data, filename):
    if not data:
        print("No data to save.")
        return

    print(f"Saving {len(data)} records to {filename}")
    df = pd.DataFrame(data)
    df.to_csv(filename, index=False, encoding='utf-8')

    print(f"Data saved to {filename}")

In [ ]:
def fetch_dataset():
    print("Starting to harvest data...")

    results = fetch_data(query='*')

    if results:
        save_data_to_csv(results, dataset_path)
    else:
        print("No data to save.")


In [ ]:
fetch_dataset()

Starting to harvest data...


Output()

Saving 917463 records to /content/drive/MyDrive/TUD-Research_Project/data/geodata_dataset.csv
Data saved to /content/drive/MyDrive/TUD-Research_Project/data/geodata_dataset.csv
